# Convert Image Frames to Video

This notebook converts sequential image frames into a video file using OpenCV.

In [12]:
import cv2
import os
from pathlib import Path
import re
from IPython.display import Video

In [13]:
def natural_sort_key(s):
    """Sort strings containing numbers in natural order."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', str(s))]

In [14]:
def create_video_from_frames(input_folder, output_file, fps=30, codec='mp4v'):
    """
    Create a video from sequential image frames.
    
    Args:
        input_folder: Path to folder containing image frames
        output_file: Path to output video file
        fps: Frames per second for the output video
        codec: Video codec to use (default: 'mp4v' for MP4)
    """
    # Get all image files from the folder
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    image_files = []
    
    input_path = Path(input_folder)
    for ext in image_extensions:
        image_files.extend(input_path.glob(f'*{ext}'))
        image_files.extend(input_path.glob(f'*{ext.upper()}'))
    
    if not image_files:
        raise ValueError(f"No image files found in {input_folder}")
    
    # Sort files naturally (e.g., frame1.png, frame2.png, ..., frame10.png)
    image_files = sorted(image_files, key=natural_sort_key)
    
    print(f"Found {len(image_files)} frames")
    print(f"First frame: {image_files[0].name}")
    print(f"Last frame: {image_files[-1].name}")
    
    # Read the first image to get dimensions
    first_frame = cv2.imread(str(image_files[0]))
    if first_frame is None:
        raise ValueError(f"Could not read first frame: {image_files[0]}")
    
    height, width, _ = first_frame.shape
    print(f"Video dimensions: {width}x{height}")
    print(f"Frame rate: {fps} fps")
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*codec)
    out = cv2.VideoWriter(output_file, fourcc, fps, (width, height))
    
    if not out.isOpened():
        raise RuntimeError("Failed to create video writer")
    
    # Write frames to video
    for i, image_file in enumerate(image_files):
        frame = cv2.imread(str(image_file))
        
        if frame is None:
            print(f"Warning: Could not read frame {image_file}, skipping...")
            continue
        
        # Resize frame if dimensions don't match
        if frame.shape[0] != height or frame.shape[1] != width:
            print(f"Warning: Frame {image_file} has different dimensions, resizing...")
            frame = cv2.resize(frame, (width, height))
        
        out.write(frame)
        
        # Progress indicator
        if (i + 1) % 10 == 0 or i == len(image_files) - 1:
            print(f"Processed {i + 1}/{len(image_files)} frames", end='\r')
    
    print()  # New line after progress
    out.release()
    print(f"Video saved to: {output_file}")
    return output_file

## Usage Example

Modify the parameters below to match your needs:

In [23]:
# Configuration
INPUT_FOLDER = "/home/saber/workspaces/irwin_ws/fyp/main/renders/torpedo_251202_perspective/seathru-nerf-lite/all_gt-rgb"  # Folder containing your image frames
OUTPUT_FILE = "/home/saber/workspaces/irwin_ws/fyp/main/renders/torpedo_251202_perspective/seathru-nerf-lite/videos/all_gt-rgb.mp4"  # Output video file
FPS = 30 # Frames per second
CODEC = 'mp4v'  # Video codec (mp4v, XVID, H264, etc.)

# Create the video
video_path = create_video_from_frames(INPUT_FOLDER, OUTPUT_FILE, FPS, CODEC)

Found 442 frames
First frame: frame_00001.jpg
Last frame: frame_00442.jpg
Video dimensions: 1232x816
Frame rate: 30 fps
Processed 442/442 frames
Video saved to: /home/saber/workspaces/irwin_ws/fyp/main/renders/torpedo_251202_perspective/seathru-nerf-lite/videos/all_gt-rgb.mp4


## Display the Video (optional)

You can display the created video directly in the notebook:

In [ ]:
# Display the video in notebook
Video(OUTPUT_FILE, width=640)